# Final_04_validation.ipynb 步驟目錄
---
| Step | 內容 | 使用資料 | 輸出檔案 |
|---:|---|---|---|
| 1 | 讀取淹水驗證資料 | `Taipei_grid_full_risk.geojson`, `易淹水.gpkg`, `78.8mm_flooding.gpkg`, `100mm_flooding.gpkg`, `130mm_flooding.gpkg` | - |
| 2 | 將外部驗證圖層套疊至 grid | `g`, `easy_flooding`, `flood_78`, `flood_100`, `flood_130` | - |
| 3 | 建立淹水驗證風險指標 | `flooding_easy_grid`, `flood_78_grid`, `flood_100_grid`, `flood_130_grid` | - |
| 4 | 進行淹水風險模型驗證 | `validation_grid`, `total_flood_risk`, `validation_flood_risk` | `output/confusion_matrix_validation.png`, `output/threshold_performance_validation.png` |
| 5 | 建立淹水風險驗證地圖 | `validation_grid`, `TOWN_MOI_1140318.shp` | `output/flood_risk_validation_confusion_map.html` |

## 淹水驗證資料準備
---
此步驟準備淹水風險驗證所需的空間資料。目的為比較本研究建立的 grid 淹水風險結果，是否與外部淹水參考圖層及不同降雨強度情境相符。

### 所需資料

| 資料 | 檔案路徑 | 用途 |
|---|---|---|
| 風險 grid | `output/Taipei_grid_full_risk.geojson` | 本研究產生的 grid 資料，包含地形、河川、坡地與總淹水風險屬性 |
| 易淹水範圍 | `data/易淹水.gpkg` | 歷史或官方認定的易淹水區參考圖層 |
| 78.8 mm/hr 淹水情境 | `data/78.8mm_flooding.gpkg` | 用於驗證 78.8 mm/hr 降雨下的淹水影響 |
| 100 mm/hr 淹水情境 | `data/100mm_flooding.gpkg` | 用於驗證 100 mm/hr 降雨下的淹水影響 |
| 130 mm/hr 淹水情境 | `data/130mm_flooding.gpkg` | 用於驗證 130 mm/hr 降雨下的淹水影響 |
---
|       降雨強度 | 大致定位   | 為何選這個值                                                     |
| ---------: | ------ | ---------------------------------------------------------- |
| 78.8 mm/hr | 基準情境   | 臺北市雨水下水道的設計保護標準，約為 5 年重現期                                  |
|  100 mm/hr | 中度極端情境 | 臺北站約 50 年重現期，用來代表超過一般排水能力、但尚未到最極端的狀況                       |
|  130 mm/hr | 極端情境   | 接近 2015 年 6 月 14 日豪雨事件，公館站一小時累積雨量達 131.5 mm，分析上超過 200 年重現期 |

### 驗證目的

淹水驗證資料用來評估高 `total_flood_risk` 的 grid 是否與已知或模擬的淹水範圍重疊。三個降雨情境圖層，`78.8 mm/hr`、`100 mm/hr` 與 `130 mm/hr`，提供外部參考基準，用來檢查模型是否能反映降雨強度增加下的淹水風險變化。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

TARGET_CRS = "EPSG:3826"

# ===== 路徑設定 =====
grid_path = Path("output/Taipei_grid_full_risk.geojson")

easy_flood_path = Path("data/易淹水.gpkg")
flood_78_path = Path("data/78.8mm_flooding.gpkg")
flood_100_path = Path("data/100mm_flooding.gpkg")
flood_130_path = Path("data/130mm_flooding.gpkg")

required_paths = {
    "risk_grid": grid_path,
    "easy_flooding": easy_flood_path,
    "flood_78_8mm_hr": flood_78_path,
    "flood_100mm_hr": flood_100_path,
    "flood_130mm_hr": flood_130_path,
}

missing_paths = {
    name: path
    for name, path in required_paths.items()
    if not path.exists()
}

if missing_paths:
    raise FileNotFoundError(f"找不到檔案: {missing_paths}")


def read_gpkg_first_layer(path, target_crs=TARGET_CRS):
    """
    讀取 GPKG。
    若未指定 layer，預設讀取第一個 layer。
    """
    layers = gpd.list_layers(path)
    layer_name = layers["name"].iloc[0]

    gdf = gpd.read_file(path, layer=layer_name)

    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=4326, allow_override=True)

    gdf = gdf.to_crs(target_crs)
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    gdf["source_layer"] = layer_name

    return gdf


# ===== 1) 讀取原本風險 grid =====
g = gpd.read_file(grid_path).to_crs(TARGET_CRS)

# ===== 2) 讀取外部驗證資料 =====
easy_flooding = read_gpkg_first_layer(easy_flood_path)
flood_78 = read_gpkg_first_layer(flood_78_path)
flood_100 = read_gpkg_first_layer(flood_100_path)
flood_130 = read_gpkg_first_layer(flood_130_path)

# ===== 3) 加上資料來源標籤 =====
easy_flooding["validation_source"] = "easy_flooding"
flood_78["validation_source"] = "78.8mm_hr"
flood_100["validation_source"] = "100mm_hr"
flood_130["validation_source"] = "130mm_hr"

# ===== 4) 基本檢查 =====
print("Risk grid:", len(g), g.crs)
print("Easy flooding:", len(easy_flooding), easy_flooding.crs)
print("78.8 mm/hr flood:", len(flood_78), flood_78.crs)
print("100 mm/hr flood:", len(flood_100), flood_100.crs)
print("130 mm/hr flood:", len(flood_130), flood_130.crs)

print("\nGrid risk columns:")
display(g.head())

print("\nEasy flooding:")
display(easy_flooding.head())

print("\n78.8 mm/hr flooding:")
display(flood_78.head())

print("\n100 mm/hr flooding:")
display(flood_100.head())

print("\n130 mm/hr flooding:")
display(flood_130.head())

### 驗證圖層套疊至 Grid
---
本步驟將外部淹水驗證資料套疊到原本的 250m × 250m 風險 grid 上，用來判斷每個 grid 是否與外部淹水資料相交。此步驟尚未進行視覺化，主要目的是產生後續驗證分析所需的 grid 屬性資料。

| 輸出變數 | 對應驗證資料 | 新增欄位 | 說明 |
|---|---|---|---|
| `flooding_easy_grid` | `easy_flooding` | `easy_flooding_hit` | 該 grid 是否與易淹水圖層相交 |
| `flooding_easy_grid` | `easy_flooding` | `easy_flooding_area_m2` | 該 grid 與易淹水圖層的相交面積 |
| `flooding_easy_grid` | `easy_flooding` | `easy_flooding_ratio` | 相交面積占 grid 面積的比例 |
| `flood_78_grid` | `flood_78` | `flood_78_hit` | 該 grid 是否與 78.8 mm/hr 淹水範圍相交 |
| `flood_78_grid` | `flood_78` | `flood_78_area_m2` | 該 grid 與 78.8 mm/hr 淹水範圍的相交面積 |
| `flood_78_grid` | `flood_78` | `flood_78_ratio` | 相交面積占 grid 面積的比例 |
| `flood_100_grid` | `flood_100` | `flood_100_hit` | 該 grid 是否與 100 mm/hr 淹水範圍相交 |
| `flood_100_grid` | `flood_100` | `flood_100_area_m2` | 該 grid 與 100 mm/hr 淹水範圍的相交面積 |
| `flood_100_grid` | `flood_100` | `flood_100_ratio` | 相交面積占 grid 面積的比例 |
| `flood_130_grid` | `flood_130` | `flood_130_hit` | 該 grid 是否與 130 mm/hr 淹水範圍相交 |
| `flood_130_grid` | `flood_130` | `flood_130_area_m2` | 該 grid 與 130 mm/hr 淹水範圍的相交面積 |
| `flood_130_grid` | `flood_130` | `flood_130_ratio` | 相交面積占 grid 面積的比例 |

其中，`*_hit` 欄位代表是否命中外部驗證資料，`*_area_m2` 表示套疊後的相交面積，`*_ratio` 則表示相交面積占該 grid 面積的比例。後續可利用這些欄位檢查 `total_flood_risk` 是否能有效反映實際或模擬淹水風險。

In [ ]:
# ===== 將驗證圖層套疊到 grid 上 =====

def overlay_validation_to_grid(
    grid_gdf,
    validation_gdf,
    hit_col,
    area_col,
    ratio_col,
):
    """
    將外部驗證圖層套疊到 grid 上。
    每個 grid 會得到：
    - hit_col: 是否與驗證圖層相交
    - area_col: 相交面積
    - ratio_col: 相交面積 / grid 面積
    """
    grid_base = grid_gdf[["grid_id", "geometry"]].copy()
    grid_base = grid_base.to_crs(TARGET_CRS)

    validation = validation_gdf.to_crs(TARGET_CRS).copy()
    validation = validation[validation.geometry.notna() & ~validation.geometry.is_empty].copy()

    grid_base["grid_area_m2"] = grid_base.geometry.area

    inter = gpd.overlay(
        grid_base[["grid_id", "grid_area_m2", "geometry"]],
        validation[["geometry"]],
        how="intersection",
        keep_geom_type=False,
    )

    if len(inter) > 0:
        inter["validation_area_m2"] = inter.geometry.area

        area_sum = (
            inter
            .groupby("grid_id", as_index=False)["validation_area_m2"]
            .sum()
            .rename(columns={"validation_area_m2": area_col})
        )
    else:
        area_sum = pd.DataFrame({
            "grid_id": [],
            area_col: [],
        })

    out = grid_gdf.copy()
    out = out.merge(area_sum, on="grid_id", how="left")

    out[area_col] = out[area_col].fillna(0.0)
    out[hit_col] = out[area_col] > 0.01
    out[ratio_col] = out[area_col] / out.geometry.area

    return out


# ===== 1) 易淹水圖層套疊 =====
flooding_easy_grid = overlay_validation_to_grid(
    grid_gdf=g,
    validation_gdf=easy_flooding,
    hit_col="easy_flooding_hit",
    area_col="easy_flooding_area_m2",
    ratio_col="easy_flooding_ratio",
)

# ===== 2) 78.8 mm/hr 淹水圖層套疊 =====
flood_78_grid = overlay_validation_to_grid(
    grid_gdf=g,
    validation_gdf=flood_78,
    hit_col="flood_78_hit",
    area_col="flood_78_area_m2",
    ratio_col="flood_78_ratio",
)

# ===== 3) 100 mm/hr 淹水圖層套疊 =====
flood_100_grid = overlay_validation_to_grid(
    grid_gdf=g,
    validation_gdf=flood_100,
    hit_col="flood_100_hit",
    area_col="flood_100_area_m2",
    ratio_col="flood_100_ratio",
)

# ===== 4) 130 mm/hr 淹水圖層套疊 =====
flood_130_grid = overlay_validation_to_grid(
    grid_gdf=g,
    validation_gdf=flood_130,
    hit_col="flood_130_hit",
    area_col="flood_130_area_m2",
    ratio_col="flood_130_ratio",
)


# ===== 基本檢查 =====
print("Grid count:", len(g))

print("Easy flooding hit grids:", flooding_easy_grid["easy_flooding_hit"].sum())
print("78.8 mm/hr hit grids:", flood_78_grid["flood_78_hit"].sum())
print("100 mm/hr hit grids:", flood_100_grid["flood_100_hit"].sum())
print("130 mm/hr hit grids:", flood_130_grid["flood_130_hit"].sum())

display(
    flooding_easy_grid[
        [
            "grid_id",
            "total_flood_risk",
            "easy_flooding_hit",
            "easy_flooding_area_m2",
            "easy_flooding_ratio",
        ]
    ].head()
)

display(
    flood_78_grid[
        [
            "grid_id",
            "total_flood_risk",
            "flood_78_hit",
            "flood_78_area_m2",
            "flood_78_ratio",
        ]
    ].head()
)

display(
    flood_100_grid[
        [
            "grid_id",
            "total_flood_risk",
            "flood_100_hit",
            "flood_100_area_m2",
            "flood_100_ratio",
        ]
    ].head()
)

display(
    flood_130_grid[
        [
            "grid_id",
            "total_flood_risk",
            "flood_130_hit",
            "flood_130_area_m2",
            "flood_130_ratio",
        ]
    ].head()
)

### 淹水驗證風險指標
---
此步驟將四個外部淹水驗證圖層合併回主要 grid：

| 圖層 | 命中欄位 |
|---|---|
| 易淹水範圍 | `easy_flooding_hit` |
| 78.8 mm/hr 淹水情境 | `flood_78_hit` |
| 100 mm/hr 淹水情境 | `flood_100_hit` |
| 130 mm/hr 淹水情境 | `flood_130_hit` |

缺失的命中值補為 `False`，面積與比例欄位則補為 `0.0`。

若一個 grid 至少與一個外部淹水圖層重疊，則視為有外部驗證命中：

**validation_any_hit = validation_hit_count > 0**

外部驗證風險分數設定如下：

| 條件 | validation_flood_risk | 風險等級 |
|---|---:|---|
| 無命中 | 0 | safe |
| 130 mm/hr 情境下淹水 | 2 | warm |
| 100 mm/hr 情境下淹水 | 4 | slight danger |
| 78.8 mm/hr 情境下淹水 | 5 | danger |
| 位於易淹水範圍 | 6 | extreme danger |

較高風險條件會覆蓋較低風險條件。此驗證分數後續用來與模型產生的 `total_flood_risk` 進行比較。

In [ ]:
# ===== 整合四個 validation grid，建立驗證風險指標 =====

validation_grid = g.copy()

# ===== 1) 選取各驗證 grid 的欄位 =====
easy_cols = [
    "grid_id",
    "easy_flooding_hit",
    "easy_flooding_area_m2",
    "easy_flooding_ratio",
]

flood_78_cols = [
    "grid_id",
    "flood_78_hit",
    "flood_78_area_m2",
    "flood_78_ratio",
]

flood_100_cols = [
    "grid_id",
    "flood_100_hit",
    "flood_100_area_m2",
    "flood_100_ratio",
]

flood_130_cols = [
    "grid_id",
    "flood_130_hit",
    "flood_130_area_m2",
    "flood_130_ratio",
]

# ===== 2) 合併回主 grid =====
validation_grid = validation_grid.merge(
    flooding_easy_grid[easy_cols],
    on="grid_id",
    how="left",
)

validation_grid = validation_grid.merge(
    flood_78_grid[flood_78_cols],
    on="grid_id",
    how="left",
)

validation_grid = validation_grid.merge(
    flood_100_grid[flood_100_cols],
    on="grid_id",
    how="left",
)

validation_grid = validation_grid.merge(
    flood_130_grid[flood_130_cols],
    on="grid_id",
    how="left",
)

# ===== 3) 缺值補 0 / False =====
hit_cols = [
    "easy_flooding_hit",
    "flood_78_hit",
    "flood_100_hit",
    "flood_130_hit",
]

area_cols = [
    "easy_flooding_area_m2",
    "flood_78_area_m2",
    "flood_100_area_m2",
    "flood_130_area_m2",
]

ratio_cols = [
    "easy_flooding_ratio",
    "flood_78_ratio",
    "flood_100_ratio",
    "flood_130_ratio",
]

for col in hit_cols:
    validation_grid[col] = validation_grid[col].fillna(False).astype(bool)

for col in area_cols + ratio_cols:
    validation_grid[col] = validation_grid[col].fillna(0.0)

# ===== 4) 建立綜合命中欄位 =====
validation_grid["validation_hit_count"] = (
    validation_grid[hit_cols]
    .astype(int)
    .sum(axis=1)
)

validation_grid["validation_any_hit"] = validation_grid["validation_hit_count"] > 0

# ===== 5) 建立外部驗證風險等級 =====
# 規則：
# 0 = safe
# 2 = warm，只要 130 mm/hr 情境下會淹
# 4 = slight danger，只要 100 mm/hr 情境下會淹
# 5 = danger，只要 78.8 mm/hr 情境下會淹
# 6 = extreme danger，只要位於 easy_flooding 範圍

validation_grid["validation_flood_risk"] = 0
validation_grid["validation_risk_level"] = "safe"

# 130 mm/hr 會淹 -> warm
validation_grid.loc[
    validation_grid["flood_130_hit"],
    "validation_flood_risk",
] = 2

validation_grid.loc[
    validation_grid["flood_130_hit"],
    "validation_risk_level",
] = "warm"

# 100 mm/hr 會淹 -> slight danger
validation_grid.loc[
    validation_grid["flood_100_hit"],
    "validation_flood_risk",
] = 4

validation_grid.loc[
    validation_grid["flood_100_hit"],
    "validation_risk_level",
] = "slight danger"

# 78.8 mm/hr 會淹 -> danger
validation_grid.loc[
    validation_grid["flood_78_hit"],
    "validation_flood_risk",
] = 5

validation_grid.loc[
    validation_grid["flood_78_hit"],
    "validation_risk_level",
] = "danger"

# easy flooding 範圍 -> extreme danger
validation_grid.loc[
    validation_grid["easy_flooding_hit"],
    "validation_flood_risk",
] = 6

validation_grid.loc[
    validation_grid["easy_flooding_hit"],
    "validation_risk_level",
] = "extreme danger"

# ===== 6) 基本檢查 =====
print("Validation grid count:", len(validation_grid))
print("Validation any-hit grids:", validation_grid["validation_any_hit"].sum())
print(
    "Validation flood risk range:",
    validation_grid["validation_flood_risk"].min(),
    "~",
    validation_grid["validation_flood_risk"].max(),
)

display(
    validation_grid[
        [
            "grid_id",
            "total_flood_risk",
            "terrain_flood_risk",
            "river_flood_risk",
            "easy_flooding_hit",
            "flood_78_hit",
            "flood_100_hit",
            "flood_130_hit",
            "validation_hit_count",
            "validation_any_hit",
            "validation_flood_risk",
            "validation_risk_level",
        ]
    ]
    .sort_values("validation_flood_risk", ascending=False)
    .head(20)
)

### 淹水風險模型驗證
---
此步驟使用外部驗證風險指標來驗證模型產生的淹水風險結果。

驗證資料與模型結果皆透過門檻值轉換為是否危險：

| 類別 | 判定規則 |
|---|---|
| 驗證資料危險 | `validation_flood_risk >= 3` |
| 模型判定危險 | `total_flood_risk >= 3` |

接著使用混淆矩陣比較模型判斷與外部驗證資料：

| 指標 | 意義 |
|---|---|
| TP | 驗證資料為危險，模型也判定危險 |
| FN | 驗證資料為危險，但模型未判定危險 |
| FP | 驗證資料非危險，但模型判定危險 |
| TN | 驗證資料非危險，模型也判定非危險 |

模型表現以四個指標評估：

| 指標 | 意義 |
|---|---|
| Recall | 模型捕捉驗證危險區的能力 |
| Precision | 模型判定危險區的可靠程度 |
| Accuracy | 整體分類正確率 |
| F1 score | Recall 與 Precision 的綜合表現 |

混淆矩陣圖輸出為：

`output/confusion_matrix_validation.png`

此外，本研究測試 `1` 至 `6` 的不同模型危險門檻，並比較各門檻下的 Recall、Precision、Accuracy 與 F1 score。

門檻敏感度分析圖輸出為：

`output/threshold_performance_validation.png`

最終採用的模型危險門檻為：

**total_flood_risk >= 3**

In [ ]:
# ===== Model validation: confusion matrix + threshold performance =====

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ===== 1) 設定 truth / model 門檻 =====
# validation_flood_risk:
# 0 = safe
# 1-2 = warm
# 3-4 = slight danger
# 5 = danger
# 6 = extreme danger
TRUTH_THRESHOLD = 3
MODEL_THRESHOLD = 3

cm_df = validation_grid.copy()

cm_df["truth_danger"] = cm_df["validation_flood_risk"] >= TRUTH_THRESHOLD
cm_df["model_danger"] = cm_df["total_flood_risk"] >= MODEL_THRESHOLD

# ===== 2) Confusion matrix =====
confusion_matrix = pd.crosstab(
    cm_df["truth_danger"],
    cm_df["model_danger"],
    rownames=["Truth danger"],
    colnames=["Model danger"],
    dropna=False,
)

confusion_matrix = confusion_matrix.reindex(
    index=[False, True],
    columns=[False, True],
    fill_value=0,
)

display(confusion_matrix)

TN = confusion_matrix.loc[False, False]
FP = confusion_matrix.loc[False, True]
FN = confusion_matrix.loc[True, False]
TP = confusion_matrix.loc[True, True]

recall = TP / (TP + FN) if (TP + FN) > 0 else np.nan
precision = TP / (TP + FP) if (TP + FP) > 0 else np.nan
accuracy = (TP + TN) / (TP + TN + FP + FN)
f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0
    else np.nan
)

metrics = pd.DataFrame({
    "metric": [
        "TP: truth danger and model danger",
        "FN: truth danger but model not danger",
        "FP: truth not danger but model danger",
        "TN: truth not danger and model not danger",
        "Recall / Sensitivity",
        "Precision",
        "Accuracy",
        "F1 score",
    ],
    "value": [
        TP,
        FN,
        FP,
        TN,
        recall,
        precision,
        accuracy,
        f1,
    ],
})

display(metrics)

# ===== 3) Confusion matrix figure =====
cm_values = np.array([
    [TN, FP],
    [FN, TP],
])

fig, ax = plt.subplots(figsize=(7.5, 6.5))

im = ax.imshow(cm_values, cmap="Blues")

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Model: Safe", "Model: Danger"])
ax.set_yticklabels(["Truth: Safe", "Truth: Danger"])

ax.set_xlabel("Model prediction", labelpad=10)
ax.set_ylabel("Truth label", labelpad=10)
ax.set_title(
    f"Confusion Matrix\nTruth threshold >= {TRUTH_THRESHOLD}, Model threshold >= {MODEL_THRESHOLD}",
    pad=14,
)

for i in range(2):
    for j in range(2):
        value = cm_values[i, j]
        ax.text(
            j,
            i,
            f"{value:,}",
            ha="center",
            va="center",
            color="white" if value > cm_values.max() / 2 else "black",
            fontsize=16,
            fontweight="bold",
        )

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.ax.set_ylabel("Grid count", rotation=270, labelpad=14)

metric_text = (
    f"Recall: {recall:.3f}    "
    f"Precision: {precision:.3f}    "
    f"Accuracy: {accuracy:.3f}    "
    f"F1: {f1:.3f}"
)

fig.text(
    0.5,
    0.03,
    metric_text,
    ha="center",
    va="center",
    fontsize=11,
    bbox=dict(
        boxstyle="round,pad=0.45",
        facecolor="white",
        edgecolor="#999999",
    ),
)

plt.subplots_adjust(bottom=0.18, right=0.88)

confusion_out_path = "output/confusion_matrix_validation.png"
plt.savefig(confusion_out_path, dpi=300, bbox_inches="tight")

print("Saved:", confusion_out_path)

plt.show()

# ===== 4) Threshold sensitivity analysis =====
truth_danger = validation_grid["validation_flood_risk"] >= TRUTH_THRESHOLD

threshold_results = []

for model_threshold in range(1, 7):
    model_danger = validation_grid["total_flood_risk"] >= model_threshold

    TP_t = ((truth_danger == True) & (model_danger == True)).sum()
    FN_t = ((truth_danger == True) & (model_danger == False)).sum()
    FP_t = ((truth_danger == False) & (model_danger == True)).sum()
    TN_t = ((truth_danger == False) & (model_danger == False)).sum()

    recall_t = TP_t / (TP_t + FN_t) if (TP_t + FN_t) > 0 else np.nan
    precision_t = TP_t / (TP_t + FP_t) if (TP_t + FP_t) > 0 else np.nan
    accuracy_t = (TP_t + TN_t) / (TP_t + TN_t + FP_t + FN_t)
    f1_t = (
        2 * precision_t * recall_t / (precision_t + recall_t)
        if (precision_t + recall_t) > 0
        else np.nan
    )

    threshold_results.append({
        "model_threshold": model_threshold,
        "TP": TP_t,
        "FN": FN_t,
        "FP": FP_t,
        "TN": TN_t,
        "recall": recall_t,
        "precision": precision_t,
        "accuracy": accuracy_t,
        "f1": f1_t,
    })

threshold_result_df = pd.DataFrame(threshold_results)

display(threshold_result_df)

# ===== 5) Threshold performance figure =====
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    threshold_result_df["model_threshold"],
    threshold_result_df["recall"],
    marker="o",
    linewidth=2,
    label="Recall",
)

ax.plot(
    threshold_result_df["model_threshold"],
    threshold_result_df["precision"],
    marker="o",
    linewidth=2,
    label="Precision",
)

ax.plot(
    threshold_result_df["model_threshold"],
    threshold_result_df["f1"],
    marker="o",
    linewidth=2,
    label="F1 score",
)

ax.plot(
    threshold_result_df["model_threshold"],
    threshold_result_df["accuracy"],
    marker="o",
    linewidth=2,
    label="Accuracy",
)

ax.axvline(
    MODEL_THRESHOLD,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label=f"Selected threshold = {MODEL_THRESHOLD}",
)

ax.set_xlabel("Model danger threshold")
ax.set_ylabel("Score")
ax.set_title("Threshold Performance for Flood Risk Validation")
ax.set_xticks(threshold_result_df["model_threshold"])
ax.set_ylim(0, 1.05)
ax.grid(True, linestyle="--", alpha=0.35)
ax.legend(loc="best")

plt.tight_layout()

threshold_out_path = "output/threshold_performance_validation.png"
plt.savefig(threshold_out_path, dpi=300, bbox_inches="tight")

print("Saved:", threshold_out_path)

plt.show()

print(
    f"Validation conclusion: model total_flood_risk >= {MODEL_THRESHOLD} "
    f"is used as the calibrated danger threshold for identifying "
    f"truth validation_flood_risk >= {TRUTH_THRESHOLD} areas."
)

### 淹水風險驗證地圖
---
此步驟將模型結果與外部驗證資料進行空間比對，並用地圖呈現 TP、FN、FP、TN 四種分類結果。

驗證門檻設定如下：

| 類別 | 判定規則 |
|---|---|
| Truth danger | `validation_flood_risk >= 3` |
| Model danger | `total_flood_risk >= 3` |

混淆矩陣分類方式：

| 類型 | 說明 |
|---|---|
| TP | 驗證資料為危險，模型也判定危險 |
| FN | 驗證資料為危險，但模型未判定危險 |
| FP | 驗證資料非危險，但模型判定危險 |
| TN | 驗證資料非危險，模型也判定非危險 |

地圖使用不同顏色呈現各類型 grid，並加入臺北市行政區邊界與 grid 邊界，方便檢查模型判斷與外部驗證資料的空間差異。

輸出檔案：

`output/flood_risk_validation_confusion_map.html`

In [ ]:
import folium
import branca.colormap as cm

# ===== Confusion map setting =====
TRUTH_THRESHOLD = 3
MODEL_THRESHOLD = 3

confusion_grid = validation_grid.copy()

confusion_grid["truth_danger"] = (
    confusion_grid["validation_flood_risk"] >= TRUTH_THRESHOLD
)

confusion_grid["model_danger"] = (
    confusion_grid["total_flood_risk"] >= MODEL_THRESHOLD
)

# ===== 建立 TP / FN / FP / TN 類別 =====
confusion_grid["confusion_type"] = "TN"

confusion_grid.loc[
    (~confusion_grid["truth_danger"]) & (confusion_grid["model_danger"]),
    "confusion_type",
] = "FP"

confusion_grid.loc[
    (confusion_grid["truth_danger"]) & (~confusion_grid["model_danger"]),
    "confusion_type",
] = "FN"

confusion_grid.loc[
    (confusion_grid["truth_danger"]) & (confusion_grid["model_danger"]),
    "confusion_type",
] = "TP"

# ===== 顏色設定 =====
confusion_color = {
    "TP": "#1a9850",  # 正確抓到危險
    "FN": "#d73027",  # 漏判，最需要檢查
    "FP": "#fdae61",  # 高估
    "TN": "#d9d9d9",  # 正確判安全
}

confusion_label = {
    "TP": "TP: truth danger and model danger",
    "FN": "FN: truth danger but model safe",
    "FP": "FP: truth safe but model danger",
    "TN": "TN: truth safe and model safe",
}

# ===== 基本統計 =====
confusion_summary = (
    confusion_grid["confusion_type"]
    .value_counts()
    .reindex(["TP", "FN", "FP", "TN"])
    .fillna(0)
    .astype(int)
    .reset_index()
)

confusion_summary.columns = ["confusion_type", "grid_count"]

display(confusion_summary)

# ===== 轉 WGS84 =====
confusion_wgs84 = confusion_grid.to_crs(epsg=4326).copy()

# ===== 建立地圖 =====
minx, miny, maxx, maxy = confusion_wgs84.total_bounds
center_lat = (miny + maxy) / 2
center_lon = (minx + maxx) / 2

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    tiles="cartodbpositron",
)

# ===== 分類圖層 =====
for ctype in ["TP", "FN", "FP", "TN"]:
    sub = confusion_wgs84[
        confusion_wgs84["confusion_type"] == ctype
    ].copy()

    if sub.empty:
        continue

    fg = folium.FeatureGroup(
        name=confusion_label[ctype],
        show=True if ctype in ["TP", "FN"] else False,
    )

    folium.GeoJson(
        sub[
            [
                "grid_id",
                "total_flood_risk",
                "validation_flood_risk",
                "validation_risk_level",
                "truth_danger",
                "model_danger",
                "confusion_type",
                "geometry",
            ]
        ].to_json(),
        style_function=lambda feature, color=confusion_color[ctype]: {
            "fillColor": color,
            "color": "#333333",
            "weight": 0.25,
            "fillOpacity": 0.65,
            "opacity": 0.45,
        },
        highlight_function=lambda feature: {
            "color": "#000000",
            "weight": 1.2,
            "fillOpacity": 0.85,
            "opacity": 1.0,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=[
                "grid_id",
                "confusion_type",
                "total_flood_risk",
                "validation_flood_risk",
                "validation_risk_level",
                "truth_danger",
                "model_danger",
            ],
            aliases=[
                "Grid ID",
                "Confusion type",
                "Model flood risk",
                "Truth flood risk",
                "Truth risk level",
                "Truth danger",
                "Model danger",
            ],
            localize=True,
            labels=True,
            sticky=False,
        ),
    ).add_to(fg)

    fg.add_to(m)

# ===== Grid boundary layer =====
grid_boundary_fg = folium.FeatureGroup(
    name="Grid Boundary",
    show=False,
)

folium.GeoJson(
    confusion_wgs84[["grid_id", "geometry"]].to_json(),
    style_function=lambda feature: {
        "fillOpacity": 0,
        "color": "#555555",
        "weight": 0.35,
        "opacity": 0.55,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["grid_id"],
        aliases=["Grid ID"],
        labels=True,
        sticky=False,
    ),
).add_to(grid_boundary_fg)

grid_boundary_fg.add_to(m)  

# ===== 讀取臺北市行政區界線 =====
town_path = "data/TOWN_MOI_1140318/TOWN_MOI_1140318.shp"

town = gpd.read_file(town_path)
town.columns = [str(c).strip().strip("'").strip('"') for c in town.columns]

if "COUNTYCODE" in town.columns:
    taipei_town = town[
        town["COUNTYCODE"].astype(str).str.contains("63000", na=False)
    ].copy()
elif "COUNTYNAME" in town.columns:
    taipei_town = town[
        town["COUNTYNAME"].astype(str).str.contains("臺北市|台北市", na=False, regex=True)
    ].copy()
else:
    taipei_town = town.copy()

taipei_town_wgs84 = taipei_town.to_crs(epsg=4326)

# ===== 臺北市行政區界線圖層 =====
town_fields = [
    c for c in ["COUNTYNAME", "TOWNNAME", "COUNTYCODE", "TOWNCODE"]
    if c in taipei_town_wgs84.columns
]

town_fg = folium.FeatureGroup(
    name="Taipei District Boundary",
    show=True,
)

folium.GeoJson(
    taipei_town_wgs84[town_fields + ["geometry"]].to_json(),
    style_function=lambda feature: {
        "color": "#111111",
        "weight": 1.5,
        "fillOpacity": 0,
        "opacity": 0.9,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=town_fields,
        aliases=town_fields,
        labels=True,
        sticky=False,
    ) if town_fields else None,
).add_to(town_fg)

town_fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

map_path = "output/flood_risk_validation_confusion_map.html"
m.save(map_path)

print("Saved:", map_path)

m